# Tiptoi Colab Pipeline v20

## GPU-Kosten & Geschwindigkeit (Whisper large-v3-turbo, ~1000 OGGs)
| GPU | CU/Stunde | ~Zeit | Kosten gesamt | Empfehlung |
|-----|-----------|-------|---------------|------------|
| T4 | ~1.6 CU/h | ~30 Min | **~0.8 CU** | ✓ Günstigste Option |
| L4 | ~3.0 CU/h | ~20 Min | ~1.0 CU | schneller, aber teurer |
| A100 40GB | ~5.4 CU/h | ~25 Min* | ~2.3 CU | kaum schneller als T4 |
| A100 80GB | ~7.5 CU/h | ~22 Min* | ~2.8 CU | teurer |
| G4 (RTX PRO 6000, 96GB) | ~8.7 CU/h | ~15 Min | ~2.2 CU | Blackwell-Arch. |
| H100 | unbekannt | ~15 Min | ? | Rate nicht dokumentiert |

\* Whisper ist **nicht GPU-limitiert** – T4 und A100 fast gleich schnell (Benchmark: 794s vs 710s für 2h Audio)

→ **T4 wählen:** günstigste Option, Whisper profitiert kaum von stärkeren GPUs

---

**Reihenfolge:**
1. OGG-ZIP entpacken → `ogg/`
2. Pyannote VAD + Speaker Embeddings → `speakers.json`
3. Whisper `large-v3-turbo` Transkription → ZIP → Drive

**Konfiguration:** Wird automatisch aus `gdrive:tiptoi/colab_config.py` geladen –
kein manuelles Eintragen nötig. Upload via `./colab_sync.sh upload`.

**Output auf Google Drive (`gdrive:tiptoi/`):**
- `{BOOK}_transcripts.zip` (Transkripte + Timestamps + speakers.json)

**Resume:** Bereits vorhandene TXT-Dateien und speakers.json werden übersprungen.

**CPU-Fallback:** Läuft automatisch auf CPU wenn GPU nicht verfügbar oder CUDA-Fehler.


## Reihenfolge beim Start
1. **Nur Zelle 0 ausführen** → installiert Pakete + startet Runtime neu
2. Nach dem Neustart: **Strg+F9** → alle Zellen laufen durch (Zelle 0 erkennt Pakete und überspringt den Neustart)


In [ ]:
# ── Schritt 0: GPU-Check + Pakete installieren ───────────────────
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Keine GPU! Laufzeit → Laufzeittyp ändern → T4 GPU → neu starten')
print(f'GPU: {torch.cuda.get_device_name(0)} ✓')

try:
    import faster_whisper, pyannote.audio
    print('✓ Pakete bereits installiert – kein Neustart nötig')
except ImportError:
    print('Installiere Pakete ...')
    import subprocess
    subprocess.run(['pip', 'install', 'faster-whisper', 'pyannote.audio', 'soundfile', 'scikit-learn', '-q'])
    print('✓ Fertig – Runtime wird neu gestartet ...')
    import os
    os.kill(os.getpid(), 9)


In [ ]:
# ── Google Drive einbinden ───────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Konfiguration – automatisch aus colab_config.py laden ───────
# colab_config.py wurde von colab_sync.sh nach gdrive:tiptoi/ hochgeladen

TIPTOI_DIR  = '/content/drive/MyDrive/tiptoi'
CONFIG_PATH = f'{TIPTOI_DIR}/colab_config.py'

exec(open(CONFIG_PATH).read())  # lädt: BOOK, VERSION, LANG

HF_TOKEN = 'YOUR_HF_TOKEN_HERE'  # HuggingFace Token → https://huggingface.co/settings/tokens

DRIVE_BASE     = f'{TIPTOI_DIR}/{BOOK}'
OGG_DIR        = f'{DRIVE_BASE}/ogg'
TRANSCRIPT_DIR = f'{DRIVE_BASE}/transcripts'
SPEAKERS_FILE  = f'{DRIVE_BASE}/speakers.json'
OGG_ZIP        = f'{TIPTOI_DIR}/{BOOK}_ogg.zip'
TRANSCRIPT_ZIP = f'{TIPTOI_DIR}/{BOOK}_transcripts.zip'

WHISPER_MODEL  = 'large-v3-turbo'
WHISPER_LANG   = 'de'
MAX_K          = 10
MIN_K          = 2   # Mindestanzahl Sprecher (auch wenn Silhouette K=2 bevorzugt)
MIN_SPEECH_SEC = 0.3
VAD_THRESHOLD  = 0.5

# ── GPU-Typ automatisch erkennen ─────────────────────────────────
def _detect_gpu_type():
    if not torch.cuda.is_available():
        return 'T4'  # Fallback
    name = torch.cuda.get_device_name(0).upper()
    if 'T4' in name:   return 'T4'
    if 'L4' in name:   return 'L4'
    if 'A100' in name:
        mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        return 'A100_80' if mem_gb > 60 else 'A100_40'
    if any(x in name for x in ('RTX', 'ADA', 'PRO 6000')):  return 'G4'
    if 'H100' in name: return 'H100'
    return 'T4'  # Unknown GPU → T4-Fallback
GPU_TYPE = _detect_gpu_type()
print(f'  GPU-Typ erkannt: {GPU_TYPE}  ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')


# ── Kostenberechnung ──────────────────────────────────────────────
import os
os.makedirs(OGG_DIR, exist_ok=True)
os.makedirs(TRANSCRIPT_DIR, exist_ok=True)

_GPU_TABLE = {
    'T4':      {'cu_h': 1.6,  'speedup': 1.0,  'label': 'NVIDIA T4 (16 GB)'},
    'L4':      {'cu_h': 3.0,  'speedup': 1.5,  'label': 'NVIDIA L4 (24 GB)'},
    'A100_40': {'cu_h': 5.4,  'speedup': 1.1,  'label': 'NVIDIA A100 40 GB'},
    'A100_80': {'cu_h': 7.5,  'speedup': 1.15, 'label': 'NVIDIA A100 80 GB'},
    'G4':      {'cu_h': 8.71, 'speedup': 2.0,  'label': 'NVIDIA RTX PRO 6000 / G4 (96 GB)'},
    'H100':    {'cu_h': None, 'speedup': 2.5,  'label': 'NVIDIA H100 (Rate unbekannt)'},
}

_BASE_MIN_PER_1000 = 30   # T4-Referenz: ~30 Min für 1000 OGGs

if GPU_TYPE not in _GPU_TABLE:
    raise ValueError(f'Unbekannter GPU_TYPE: {GPU_TYPE}. Erlaubt: {list(_GPU_TABLE)}')

_g = _GPU_TABLE[GPU_TYPE]
_ogg_count = len([f for f in __import__('pathlib').Path(OGG_DIR).glob('*.ogg')]) if __import__('pathlib').Path(OGG_DIR).exists() else None
_est_min   = _BASE_MIN_PER_1000 / _g['speedup']
_est_h     = _est_min / 60
_est_cu    = round(_est_h * _g['cu_h'], 2) if _g['cu_h'] else None

print(f'✓ Config geladen: {CONFIG_PATH}')
print(f'  Buch:    {BOOK}')
print(f'  Version: {VERSION}  |  Sprache: {LANG}')
print()
print(f'  GPU:     {_g["label"]}')
if _ogg_count:
    _est_min_real = (_BASE_MIN_PER_1000 / _g['speedup']) * (_ogg_count / 1000)
    _est_cu_real  = round((_est_min_real / 60) * _g['cu_h'], 2) if _g['cu_h'] else '?'
    print(f'  OGGs:    {_ogg_count} Dateien')
    print(f'  Geschätzte Zeit:   ~{_est_min_real:.0f} Min')
    print(f'  Geschätzte Kosten: ~{_est_cu_real} CU  (~{round(_est_cu_real * 0.10, 2) if isinstance(_est_cu_real, float) else "?"}€)')
else:
    print(f'  (OGG-Ordner noch leer – Schätzung nach Entpacken)')
    print(f'  Basis (1000 OGGs): ~{_est_min:.0f} Min  |  ~{_est_cu if _est_cu else "?"} CU')

In [ ]:
# ── Schritt 0b: OGG-ZIP entpacken ────────────────────────────────
import zipfile, os

ogg_count_before = len(list(__import__('pathlib').Path(OGG_DIR).glob('*.ogg')))

if ogg_count_before > 0:
    print(f'{ogg_count_before} OGGs bereits vorhanden → Entpacken übersprungen.')
else:
    print(f'Entpacke {OGG_ZIP} → {OGG_DIR} ...')
    with zipfile.ZipFile(OGG_ZIP) as zf:
        zf.extractall(OGG_DIR)
    ogg_count = len(list(__import__('pathlib').Path(OGG_DIR).glob('*.ogg')))
    print(f'✓ {ogg_count} OGGs entpackt.')


In [ ]:
# ── Schritt 1a: Pyannote Modelle laden ──────────────────────────
import torch
import soundfile as sf
import numpy as np
from pathlib import Path
from pyannote.audio import Inference, Model
from huggingface_hub import login

login(token=HF_TOKEN)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

print('Lade VAD-Modell (pyannote/segmentation-3.0)...')
vad_model = Inference(
    Model.from_pretrained('pyannote/segmentation-3.0'),
    window='whole', device=device
)
print('VAD geladen ✓')

print('Lade Embedding-Modell (pyannote/embedding)...')
emb_model = Inference(
    Model.from_pretrained('pyannote/embedding'),
    window='whole', device=device
)
print('Embedding geladen ✓')

In [ ]:
# ── Schritt 1b: Resume-Check (speakers.json vorhanden?) ──────────
import json

if os.path.exists(SPEAKERS_FILE):
    print(f'speakers.json bereits vorhanden → Sprecher-Analyse wird übersprungen.')
    with open(SPEAKERS_FILE) as f:
        spk_data = json.load(f)
    print(f'  Sprecher: {spk_data["num_speakers"]}  |  '
          f'Sprache: {spk_data.get("speech_count","?")}  |  '
          f'Geräusch: {spk_data.get("noise_count","?")}  |  '
          f'Gemischt: {spk_data.get("mixed_count","?")}')
    RUN_PYANNOTE = False
else:
    RUN_PYANNOTE = True
    print('speakers.json nicht gefunden → Sprecher-Analyse läuft.')

In [ ]:
# ── Schritt 1b: VAD + Embeddings extrahieren ─────────────────────
if RUN_PYANNOTE:
    ogg_files = sorted(Path(OGG_DIR).glob('*.ogg'))
    print(f'{len(ogg_files)} OGGs gefunden')

    records      = []   # (stem, embedding | None) – in Reihenfolge
    noise_count  = 0
    mixed_count  = 0
    speech_count = 0
    emb_size     = None

    for i, ogg in enumerate(ogg_files):
        audio, sr = sf.read(str(ogg))
        if audio.ndim > 1:
            audio = audio.mean(axis=1)

        waveform    = torch.tensor(audio, dtype=torch.float32).unsqueeze(0)
        audio_input = {'waveform': waveform, 'sample_rate': sr}

        # VAD: speech-Ratio berechnen
        try:
            vad_raw    = vad_model(audio_input)
            vad_data   = np.asarray(vad_raw.data if hasattr(vad_raw, 'data') else vad_raw)
            speech_prob = vad_data.max(axis=-1) if vad_data.ndim > 1 else vad_data.flatten()
        except Exception as e:
            print(f'  ⚠ VAD-Fehler bei {ogg.name}: {e}')
            records.append((ogg.stem, None))
            noise_count += 1
            continue

        speech_ratio = float((speech_prob >= VAD_THRESHOLD).mean())
        speech_dur   = speech_ratio * len(audio) / sr

        if speech_dur < MIN_SPEECH_SEC:
            noise_count += 1
            records.append((ogg.stem, None))
            continue

        if speech_ratio < 0.8:
            mixed_count += 1
        else:
            speech_count += 1

        try:
            emb      = np.asarray(emb_model(audio_input)).flatten()
            emb_size = len(emb)
            records.append((ogg.stem, emb))
        except Exception as e:
            print(f'  ⚠ Embedding-Fehler bei {ogg.name}: {e}')
            records.append((ogg.stem, None))
            noise_count += 1
            speech_count -= 1

        if (i + 1) % 200 == 0 or (i + 1) == len(ogg_files):
            print(f'[{i+1}/{len(ogg_files)}] '
                  f'Sprache={speech_count} Geräusch={noise_count} Gemischt={mixed_count}')

    # Einheitliche Listen aufbauen – NaN für Geräusche/Fehler
    if emb_size is None:
        emb_size = 512
    stems      = [r[0] for r in records]
    embeddings = [r[1] if r[1] is not None else np.full(emb_size, np.nan) for r in records]

    print(''); print(f'Fertig! Sprache={speech_count} · Geräusch={noise_count} · Gemischt={mixed_count}')
else:
    print('RUN_PYANNOTE=False → übersprungen.')

In [ ]:
# ── Schritt 1c: Clustering → speakers.json ───────────────────────
if RUN_PYANNOTE:
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score
    from sklearn.preprocessing import normalize
    from collections import Counter

    X = np.vstack(embeddings)

    valid        = ~np.isnan(X).any(axis=1)
    X_clean      = X[valid]
    valid_stems  = [s for s, v in zip(stems, valid) if v]
    nan_count    = int((~valid).sum())
    if nan_count > 0:
        print(f'  {nan_count} Embeddings mit NaN entfernt')
    print(f'  {len(valid_stems)} gültige Embeddings für Clustering')

    best_k, best_score, best_labels = 1, -1.0, None
    all_scores = {}  # K → Silhouette-Score (alle getesteten K)

    if len(valid_stems) < 2:
        print(f'  ⚠ Zu wenig gültige Samples ({len(valid_stems)}) – Clustering übersprungen.')
        best_k = 1
        best_labels = [0] * len(valid_stems)
    else:
        X_norm = normalize(X_clean)
        for k in range(2, min(MAX_K, len(valid_stems) - 1) + 1):
            km     = KMeans(n_clusters=k, random_state=42, n_init=10)
            labels = km.fit_predict(X_norm)
            score  = silhouette_score(X_norm, labels)
            all_scores[k] = round(float(score), 4)
            print(f'  K={k}: Silhouette={score:.4f}')
            if score > best_score:
                best_score, best_k, best_labels = score, k, labels

        if best_labels is None:
            best_labels = [0] * len(valid_stems)

    # MIN_K erzwingen: falls bestes K < MIN_K, stattdessen MIN_K verwenden
    if best_k < MIN_K and MIN_K in all_scores:
        km_forced = KMeans(n_clusters=MIN_K, random_state=42, n_init=10).fit(X_norm)
        best_labels = km_forced.labels_
        print(f'  ⚠ Silhouette-Optimum K={best_k} < MIN_K={MIN_K} → erzwinge K={MIN_K}')
        best_k = MIN_K
    print(f'\n→ Bestes K: {best_k}  (Silhouette: {best_score:.4f})')

    voice_map = {stem: int(lbl) for stem, lbl in zip(valid_stems, best_labels)}
    spk_data  = {
        'num_speakers':     best_k,
        'silhouette':       round(float(best_score), 4),
        'silhouette_scores': all_scores,
        'noise_count':      noise_count,
        'mixed_count':      mixed_count,
        'speech_count':     speech_count,
        'voice_map':        voice_map
    }
    with open(SPEAKERS_FILE, 'w', encoding='utf-8') as f:
        json.dump(spk_data, f, indent=2, ensure_ascii=False)

    print(f'\nspeakers.json gespeichert ✓')
    for spk, cnt in sorted(Counter(voice_map.values()).items()):
        print(f'  Sprecher {spk}: {cnt} Dateien')
    print(f'  Geräusche: {noise_count}  Gemischt: {mixed_count}  Sprache: {speech_count}')
else:
    print('RUN_PYANNOTE=False → übersprungen.')

In [ ]:
# ── Schritt 2a: Whisper laden ────────────────────────────────────
import torch
from faster_whisper import WhisperModel
import zipfile, shutil

LOCAL_TMP = '/tmp/transcripts'
os.makedirs(LOCAL_TMP, exist_ok=True)

def _cuda_ok():
    try:
        torch.zeros(1).cuda()
        return True
    except Exception:
        return False

if torch.cuda.is_available() and _cuda_ok():
    WHISPER_DEVICE  = 'cuda'
    WHISPER_COMPUTE = 'float16'
else:
    WHISPER_DEVICE  = 'cpu'
    WHISPER_COMPUTE = 'int8'
    print(f"⚠ Kein GPU – Whisper läuft auf CPU (int8). Bei {WHISPER_MODEL} sehr langsam!")

print(f'Lade Whisper {WHISPER_MODEL} ({WHISPER_DEVICE}/{WHISPER_COMPUTE})...')
whisper = WhisperModel(WHISPER_MODEL, device=WHISPER_DEVICE, compute_type=WHISPER_COMPUTE)
print('Whisper geladen ✓')

In [ ]:
# ── Schritt 2b: Transkription + Timestamps + ZIP → Drive ─────────
import time as _time_mod
_t_start = _time_mod.time()

ogg_files = sorted(Path(OGG_DIR).glob("*.ogg"))
print(f"{len(ogg_files)} OGGs gefunden")

skipped = done = errors = 0

for i, ogg in enumerate(ogg_files):
    txt_path = Path(LOCAL_TMP) / f"{ogg.stem}_{VERSION}.txt"

    if txt_path.exists():
        skipped += 1
        continue

    try:
        segs, info = whisper.transcribe(
            str(ogg), language=WHISPER_LANG, beam_size=5,
            vad_filter=True,
            vad_parameters=dict(min_silence_duration_ms=300)
        )
        segs = list(segs)
        text = " ".join(s.text.strip() for s in segs).strip()
        txt_path.write_text(text, encoding="utf-8")

        if segs:
            import soundfile as sf_ts
            dur = sf_ts.info(str(ogg)).duration
            ts = {"start": round(segs[0].start, 3),
                  "end":   round(segs[-1].end, 3),
                  "duration": round(dur, 3)}
            ts_path = Path(LOCAL_TMP) / f"{ogg.stem}_{VERSION}_ts.json"
            ts_path.write_text(json.dumps(ts), encoding="utf-8")
        done += 1
    except Exception as e:
        txt_path.write_text("", encoding="utf-8")
        errors += 1

    if (i + 1) % 200 == 0 or (i + 1) == len(ogg_files):
        print(f"[{i+1}/{len(ogg_files)}] done={done} skip={skipped} err={errors}")

print(f"Fertig! {done} transkribiert · {skipped} übersprungen · {errors} Fehler")

# Als ZIP auf Drive hochladen (TXT + JSON + speakers.json)
print("Erstelle ZIP...")
ZIP_PATH = f"/tmp/{BOOK}_transcripts.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in Path(LOCAL_TMP).glob("*.txt"):
        zf.write(f, f.name)
    for f in Path(LOCAL_TMP).glob("*.json"):
        zf.write(f, f.name)
    # speakers.json immer mit einpacken → pipeline.py findet es nach dem Entpacken
    if os.path.exists(SPEAKERS_FILE):
        zf.write(SPEAKERS_FILE, "speakers.json")
        print("speakers.json in ZIP aufgenommen ✓")
    else:
        print("⚠ speakers.json nicht gefunden – Sprecher-Analyse hat gefehlt!")

zip_mb = os.path.getsize(ZIP_PATH) / 1024 / 1024
print(f"ZIP-Größe: {zip_mb:.1f} MB")

shutil.copy(ZIP_PATH, TRANSCRIPT_ZIP)
print(f"ZIP hochgeladen → {TRANSCRIPT_ZIP} ✓")

# ── Zeitmessung + Benchmark-Log auf Drive ────────────────────────
_t_end     = _time_mod.time()
_elapsed_s = _t_end - _t_start
_elapsed_min = _elapsed_s / 60
_ogg_total = len(ogg_files)

_g         = _GPU_TABLE.get(GPU_TYPE, {})
_cu_h      = _g.get('cu_h')
_actual_cu = round((_elapsed_s / 3600) * _cu_h, 4) if _cu_h else None
_cu_per_1000 = round(_actual_cu / _ogg_total * 1000, 4) if (_actual_cu and _ogg_total) else None
_min_per_1000 = round(_elapsed_min / _ogg_total * 1000, 1) if _ogg_total else None

print()
print(f"── Benchmark ──────────────────────────────────────")
print(f"  GPU:          {GPU_TYPE}  ({_g.get('label','')})")
print(f"  OGGs:         {_ogg_total}")
print(f"  Laufzeit:     {_elapsed_min:.1f} Min  ({_elapsed_s:.0f} s)")
print(f"  Laufzeit/1000:{_min_per_1000} Min")
if _actual_cu:
    print(f"  Verbrauch:    {_actual_cu} CU  (~{round(_actual_cu*0.10,3)}€)")
    print(f"  CU/1000 OGGs: {_cu_per_1000} CU")
else:
    print(f"  Verbrauch:    unbekannt (H100-Rate nicht dokumentiert)")

import csv, datetime
LOG_PATH = f"{TIPTOI_DIR}/gpu_benchmark.csv"
_now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
_row = {
    "datum":        _now,
    "buch":         BOOK,
    "gpu":          GPU_TYPE,
    "ogg_anzahl":   _ogg_total,
    "laufzeit_min": round(_elapsed_min, 2),
    "min_per_1000": _min_per_1000,
    "actual_cu":    _actual_cu if _actual_cu else "",
    "cu_per_1000":  _cu_per_1000 if _cu_per_1000 else "",
    "kosten_eur":   round(_actual_cu * 0.10, 4) if _actual_cu else "",
}
_fields = list(_row.keys())
_write_header = not __import__("os").path.exists(LOG_PATH)
with open(LOG_PATH, "a", newline="", encoding="utf-8") as _f:
    _w = csv.DictWriter(_f, fieldnames=_fields)
    if _write_header:
        _w.writeheader()
    _w.writerow(_row)
print(f"  → Log gespeichert: {LOG_PATH}")


In [ ]:
# ── Schritt 3: Colab-Session beenden (keine weiteren Kosten) ─────
print('Alle Schritte abgeschlossen. Session wird beendet ...')
from google.colab import runtime
runtime.unassign()